# 03 - Screening Kandidat Refusal Direction

**Tujuan:** Uji top kandidat arah dengan inference-time ablation untuk cari layer paling efektif.

**Auto-detect:** Berjalan otomatis di **local (VS Code)** maupun **Google Colab**.

---
**Output:** `models/abliterated/screening_results.csv`


Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import torch
import pandas as pd
from safetensors.torch import load_file
from transformer_lens import HookedTransformer, utils
import functools
from pathlib import Path

from src.config import DEFAULT_MODEL, DEVICE, N_INST_TRAIN, N_INST_TEST, ABLITERATED_DIR, SEED
from src.data_loader import get_harmful_instructions, get_harmless_instructions, get_train_test_split, create_holdout_set
from src.generation_utils import tokenize_instructions_fn, get_generations, direction_ablation_hook, make_ablation_hooks
from src.metrics import compute_refusal_rate, classify_refusal

torch.manual_seed(SEED)
print(f"Device: {DEVICE}")


In [ ]:
# --- Load model ---
model = HookedTransformer.from_pretrained_no_processing(
    DEFAULT_MODEL,
    dtype=torch.bfloat16,
    default_padding_side="left",
    device=DEVICE,
)
model.tokenizer.padding_side = "left"
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f"Model loaded. n_layers={model.cfg.n_layers}")


In [ ]:
# --- Load refusal directions ---
refusal_dirs = load_file(str(ABLITERATED_DIR / "refusal_dirs.safetensors"))
print(f"Loaded {len(refusal_dirs)} refusal directions:")
for k, v in refusal_dirs.items():
    print(f"  {k}: shape={v.shape}")


In [ ]:
# --- Load test prompts ---
harmful = get_harmful_instructions()
harmless = get_harmless_instructions()
holdout_harmful, holdout_harmless = create_holdout_set(harmful, harmless)
_, _, harmful_test, harmless_test = get_train_test_split(harmful, harmless, holdout_harmful, holdout_harmless, n_train=N_INST_TRAIN, n_test=N_INST_TEST)

print(f"Test prompts: {len(harmful_test)} harmful + {len(harmless_test)} harmless")

# Gunakan subset kecil untuk screening
screening_prompts = harmful_test[:8]
print(f"Screening prompts: {len(screening_prompts)}")


In [ ]:
# --- Screening ---
# Key dari safetensors: resid_pre.layer_1, resid_pre.layer_2, ...
results = []

for layer in range(1, model.cfg.n_layers):
    dir_key = f"resid_pre.layer_{layer}"
    direction = refusal_dirs.get(dir_key)
    if direction is None:
        print(f"  Layer {layer}: direction not found, skipping")
        continue

    hooks = make_ablation_hooks(model, direction, layer_indices=[layer], act_names=["resid_pre"])
    gens = get_generations(model, screening_prompts, tokenize_instructions_fn, fwd_hooks=hooks)
    rr = compute_refusal_rate(gens)
    results.append({"layer": layer, "act_name": "resid_pre", "refusal_rate": rr})
    print(f"Layer {layer}: RR = {rr:.2%}")

    # Bersihin cache
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# --- Analisis hasil ---
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("refusal_rate", ascending=True)

print("\n=== Top 5 Layer Paling Efektif (RR Terendah) ===\n")
print(df_results.head(5))

# Simpan
df_results.to_csv(ABLITERATED_DIR / "screening_results.csv", index=False)
print(f"\nSaved: {ABLITERATED_DIR / 'screening_results.csv'}")

# Layer terbaik
best_layer = int(df_results.iloc[0]["layer"])
print(f"\nLayer terbaik: {best_layer} (RR = {df_results.iloc[0]['refusal_rate']:.2%})")


In [ ]:
print(f"Screening results: {ABLITERATED_DIR / 'screening_results.csv'}")


---
**Next:** Lanjut ke `04_orthogonalize_save.ipynb` untuk apply orthogonalization permanen.
